# Resume Optimizer — QC Microbiology auto-apply pipeline

End-to-end orchestration. Run cells top-to-bottom on the first day; thereafter you can re-run from **Cell 2 (fetch)** onwards.

Stages:
1. Load config + parsed resume
2. Fetch jobs from JSearch + JobSpy (Naukri / LinkedIn / Indeed)
3. Drop ghost / dead postings
4. Rank by fit + salary + recency
5. Mine market keywords (cGMP / EM / sterility, etc.)
6. Rewrite a master ATS-clean resume
7. For each top-N job, generate a tailored resume + cover letter + screening answers
8. Push everything to the Google Sheet tracker (status=queued)
9. Walk the apply queue (assist mode)
10. Run the Gmail watcher in the background to auto-advance statuses

## 1. Setup — config, paths, resume

In [ ]:
from __future__ import annotations
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from src import config as cfg
from src import resume_parser
from src import job_fetcher, jobspy_fetcher
from src import engagement_filter, ranker, keyword_extractor
from src import resume_writer, ats_scorer
from src import cover_letter, screening_qs
from src import tracker as tracker_mod
from src import apply_queue, email_watcher

PREFS = cfg.load_preferences()
REPO_ROOT = cfg.REPO_ROOT
INPUT_RESUME = REPO_ROOT / 'data' / 'input' / 'resume.pdf'
if not INPUT_RESUME.exists():
    # try .docx then .txt
    for ext in ('.docx', '.txt'):
        cand = REPO_ROOT / 'data' / 'input' / f'resume{ext}'
        if cand.exists():
            INPUT_RESUME = cand
            break
OUTPUT_DIR = REPO_ROOT / 'data' / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
APPS_ROOT = REPO_ROOT / (PREFS.get('apply_queue', {}).get('artifacts_root') or 'data/output/applications')
APPS_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Resume path:    {INPUT_RESUME}')
print(f'Resume exists?  {INPUT_RESUME.exists()}')
print(f'Apps artifacts: {APPS_ROOT}')
print(f'Top-level prefs keys: {list(PREFS.keys())}')

In [ ]:
assert INPUT_RESUME.exists(), (
    f'Drop your resume at {INPUT_RESUME.parent}/resume.(pdf|docx|txt) before running.'
)
RESUME = resume_parser.parse_resume(INPUT_RESUME, use_llm=True)
print('Name:    ', RESUME['contact'].get('name'))
print('Email:   ', RESUME['contact'].get('email'))
print('Skills:  ', len(RESUME.get('skills', [])))
print('Roles:   ', len(RESUME.get('experience', [])))
print('Summary len:', len(RESUME.get('summary', '')))
RESUME_OUT = OUTPUT_DIR / 'resume_parsed.json'
RESUME_OUT.write_text(json.dumps(RESUME, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Saved structured resume -> {RESUME_OUT}')

## 2. Fetch jobs from both sources

In [ ]:
search = PREFS.get('search', {})
role = search.get('role', 'Senior Executive QC Microbiology')
role_aliases = [role] + list(search.get('role_aliases', []) or [])
locations = list(search.get('locations', ['Bangalore']))
country = search.get('country', 'in')

# JSearch (LinkedIn + Indeed + Google Jobs aggregator)
jsearch_query = job_fetcher.JobQuery(
    role=role,
    domain=search.get('domain', ''),
    locations=locations,
    employment_types=search.get('employment_types', 'FULLTIME'),
    date_posted=search.get('date_posted', 'week'),
    remote_only=bool(search.get('remote_only', False)),
    num_pages=2,
    country=country,
)
jsearch_jobs = job_fetcher.fetch_jobs(jsearch_query)
for j in jsearch_jobs:
    j.setdefault('_source', 'jsearch')

# JobSpy (Naukri + LinkedIn + Indeed direct)
jobspy_jobs = jobspy_fetcher.fetch_for_role_aliases(
    role_aliases,
    locations=locations,
    sites=['naukri', 'indeed', 'linkedin'],
    hours_old=24 * 14,
    results_wanted=25,
    country_indeed='India',
)

merged = engagement_filter.merge_sources(jsearch_jobs, jobspy_jobs)
print(f'JSearch: {len(jsearch_jobs)} | JobSpy: {len(jobspy_jobs)} | Merged unique: {len(merged)}')

## 3. Drop ghost / dead postings

In [ ]:
fcfg = PREFS.get('filter', {})
sel = PREFS.get('selection', {})
kept, stats = engagement_filter.filter_jobs(
    merged,
    max_age_days=fcfg.get('max_age_days', 21),
    max_applicants=fcfg.get('max_applicants', 200),
    validate_apply_link=bool(fcfg.get('validate_apply_link', False)),
    employer_blacklist=sel.get('employer_blacklist', []) or [],
    recruiter_keywords=fcfg.get('recruiter_keywords', []),
    salary_floor_lpa=sel.get('salary_floor_lpa'),
)
print(json.dumps(stats.as_dict(), indent=2))

## 4. Rank

In [ ]:
rcfg = PREFS.get('ranking', {})
weights = (
    float(rcfg.get('fit_weight', 0.45)),
    float(rcfg.get('salary_weight', 0.35)),
    float(rcfg.get('recency_weight', 0.20)),
)
ranked_df = ranker.rank_jobs(kept, RESUME, weights=weights)
TOP_N = int(sel.get('top_n_per_run', 25))
top_df = ranker.top_n(ranked_df, n=TOP_N)
display(top_df[['title', 'company', 'city', 'posted_days_ago', 'salary_mid_lpa', 'fit_score', 'composite_score']].head(15))

## 5. Mine market keywords + seed the domain

In [ ]:
jds_for_mining = top_df['description'].tolist()
seed = search.get('domain_keywords_seed', []) or []
# Inject seed keywords by appending a synthetic 'JD' so every domain pillar is at least considered.
if seed:
    jds_for_mining = jds_for_mining + ['\n'.join(seed)]
records = keyword_extractor.extract_market_keywords(
    jds_for_mining,
    role=role,
    domain=search.get('domain', ''),
)
kw_df = pd.DataFrame([
    {'tier': r.tier, 'category': r.category, 'keyword': r.display, 'freq_%': round(r.frequency*100,1), 'jds': r.raw_count}
    for r in records
])
display(kw_df.head(40))
critical = [r.display for r in records if r.tier == 'critical']
recommended = [r.display for r in records if r.tier == 'recommended']
print(f'critical={len(critical)} recommended={len(recommended)} total_kept={len(records)}')

## 6. Rewrite a master ATS-clean resume

In [ ]:
master = resume_writer.optimize_resume(
    RESUME,
    role=role,
    domain=search.get('domain', ''),
    critical_keywords=critical,
    recommended_keywords=recommended,
)
master_docx = OUTPUT_DIR / 'optimized_resume.docx'
resume_writer.write_docx(master, master_docx)
print(f'Wrote master resume -> {master_docx}')

before_report = ats_scorer.score_resume(
    resume_writer.write_docx(RESUME, OUTPUT_DIR / '_before_resume.docx'),
    records,
)
after_report = ats_scorer.score_resume(master_docx, records)
display(ats_scorer.compare_before_after(before_report, after_report))
print(f'Overall: {before_report.overall} -> {after_report.overall}')

## 7. Per-job artifacts (resume + cover letter + screening answers)

In [ ]:
from src.resume_writer import safe_filename
from src.ats_scorer import score_against_jd

tailor_threshold = float(sel.get('tailor_when_coverage_below', 0.90))
MASTER_FOR_TAILOR = master  # rewritten dict
artifacts: list[dict] = []

for _, job_row in top_df.iterrows():
    job = job_row.to_dict()
    slug = f"{safe_filename(job.get('company',''))}_{safe_filename(job.get('title',''))}"
    job_dir = APPS_ROOT / slug
    job_dir.mkdir(parents=True, exist_ok=True)

    # Decide whether to spin a tailored variant or just copy the master
    jd_keywords = [r.display for r in records if r.tier in {'critical', 'recommended'}][:25]
    jd_cov = score_against_jd(json.dumps(MASTER_FOR_TAILOR, ensure_ascii=False), job.get('description',''), jd_keywords)
    if jd_cov['coverage'] < tailor_threshold:
        tailored = resume_writer.tailor_for_job(MASTER_FOR_TAILOR, job, job_keywords=jd_keywords)
    else:
        tailored = MASTER_FOR_TAILOR
    resume_path = job_dir / 'resume.docx'
    resume_writer.write_docx(tailored, resume_path)

    # Cover letter
    cl_path = job_dir / 'cover_letter.docx'
    cover_letter.generate_and_save(
        tailored, job, cl_path,
        role=role, domain=search.get('domain',''), critical_keywords=critical,
    )

    # Screening answers
    answers = screening_qs.generate_screening_answers(tailored, job, prefs=PREFS) or {}
    screening_qs.write_answers_json(answers, job_dir / 'screening_answers.json')

    artifacts.append({
        'job_id': job['job_id'],
        'company': job.get('company',''),
        'title': job.get('title',''),
        'location': job.get('city',''),
        'apply_link': job.get('apply_link',''),
        'salary_mid_lpa': job.get('salary_mid_lpa'),
        'fit_score': job.get('fit_score'),
        'composite_score': job.get('composite_score'),
        'posted_days_ago': job.get('posted_days_ago'),
        'source': job.get('_source', 'jsearch'),
        'artifact_dir': str(job_dir),
        'jd_coverage': jd_cov['coverage'],
    })

print(f'Generated artifacts for {len(artifacts)} jobs under {APPS_ROOT}')

## 8. Push to Google Sheet tracker

In [ ]:
TRACKER = tracker_mod.open_tracker(PREFS)
for a in artifacts:
    row = tracker_mod.job_dict_to_row(
        # tracker_mod.job_dict_to_row expects the JSearch/JobSpy shape; remap
        {
            'job_id': a['job_id'],
            'employer_name': a['company'],
            'job_title': a['title'],
            'job_city': a['location'],
            'job_apply_link': a['apply_link'],
            '_source': a['source'],
        },
        artifact_dir=a['artifact_dir'],
        fit_score=a['fit_score'],
        composite_score=a['composite_score'],
        salary_mid_lpa=a['salary_mid_lpa'],
        posted_days_ago=a['posted_days_ago'],
    )
    TRACKER.upsert_job(row)
print(f'Tracker now has {len(TRACKER.list_rows())} rows.')

## 9. Walk the apply queue (assist mode)
Press `s` after you click Submit on the portal.

In [ ]:
cap = int((PREFS.get('apply_queue', {}) or {}).get('daily_apply_cap', 12))
# Preview first; switch to run_queue(...) to actually open browser tabs.
apply_queue.dry_run(TRACKER, limit=cap)

In [ ]:
# Uncomment to actually start the assist queue.
# apply_queue.run_queue(TRACKER, daily_cap=cap)

## 10. Gmail watcher — auto-advance status on HR replies
First run pops a browser for one-time OAuth consent. Subsequent runs use the cached token.

In [ ]:
ewc = PREFS.get('email_watcher', {})
email_watcher.poll_once(
    TRACKER,
    lookback_days=7,
    history_path=ewc.get('history_file', 'data/output/.gmail_history.json'),
)
# For continuous polling (blocking), uncomment:
# email_watcher.watch_forever(TRACKER, poll_interval_minutes=int(ewc.get('poll_interval_minutes', 15)))

## 11. Dashboard read-back

In [ ]:
rows = TRACKER.list_rows()
if rows:
    df = pd.DataFrame(rows)
    summary = df.groupby('status').size().rename('count').reset_index()
    display(summary)
    display(df[['company','title','location','status','applied_at','last_response_summary']].head(25))
else:
    print('Tracker is empty.')